In [ ]:
import pandas as pd 
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split
from small_text import TextDataset
import numpy as np
import small_text
import pickle

# Import preprocessed Data
Preprocessing was done in R, "data_activelearning.parquet" loads the processed corpus.

In [4]:
# Load corpus
data = pq.read_table("../data/data_activelearning.parquet")
data = data.to_pandas() \
     .query("id_subcorpus in ['psyB', 'psyC']") \
     .reset_index(drop=True)

# Train-Test Splits for each Subcorpus (n=100)

In [5]:
datasets = {}

for group, sub_df in data.groupby('id_subcorpus', observed=True): #group -> group-names, sub_df -> subset df for groups
    train, test = train_test_split(sub_df, test_size = 100, random_state=1234)
    train.loc[:, 'label'] = small_text.base.LABEL_UNLABELED


    datasets[group] = {'train': train, 'test': test}

# Initialization-Set Splitting (n=50)

In [ ]:
datasets_init = {}

for dataset in datasets: #group = group-names, sub_df = subset df for groups
    train, init = train_test_split(datasets[dataset]["train"], test_size = 50, random_state=1234)
    train.loc[:, 'label'] = small_text.base.LABEL_UNLABELED
    datasets_init[dataset] = {'train': train, 'test': datasets[dataset]["test"], 'init': init}

In [9]:
# Export test and init sets of schizophrenia and depression for labeling
datasets['psyB']['init'].to_csv("../data/schizophrenia_init.csv", index=False)
datasets['psyB']['test'].to_csv("../data/schizophrenia_test.csv", index=False)
datasets['psyC']['init'].to_csv("../data/depression_init.csv", index=False)
datasets['psyC']['test'].to_csv("../data/depression_test.csv", index=False)

In [ ]:
# Reimport the labeled init sets for schizophrenia and depression
datasets['psyB']['init'] = pd.read_csv("../data/schizophrenia_init_labeled.csv")
datasets['psyC']['init'] = pd.read_csv("../data/depression_init_labeled.csv")
datasets['psyB']['test'] = pd.read_csv("../data/schizophrenia_test_labeled.csv")
datasets['psyC']['test'] = pd.read_csv("../data/depression_test_labeled.csv")

# Correct Labels for subset training
Transform to binary labels

In [ ]:
def transform_labels(df):
    df['label'] = df['label'].astype(int)
    
    # Define conditions and corresponding choices
    conditions = [
        (df['label'] == 0),
        (df['label'] == 1),
        (df['label'] == 2)
    ]
    
    choices = [0, 1, 1]
    
    # Apply case_when equivalent
    df['label_binary'] = np.select(conditions, choices, default=-1)
    return df

# Apply the transformation to psyB and psyC (schizophrenia and depression)
for key in ['psyB', 'psyC']:
    datasets[key]['init'] = transform_labels(datasets[key]['init'])
    datasets[key]['test'] = transform_labels(datasets[key]['test'])
    datasets[key]['train']['label_binary'] = datasets[key]['train']['label']

In [ ]:
# Combine init and train 
for dataset in datasets:
    datasets[dataset]['init_plus_train'] = pd.concat([datasets[dataset]['init'], datasets[dataset]['train']], ignore_index=True)

In [8]:
# Save datasets at current state
with open("../../data/datasets.pkl", 'wb') as f:
    pickle.dump(datasets, f)


# Transform to and save as Small-Text Datasets
Both datasets are transformed to work with the small-text active learning library, and saved as "small_text_datasets.pkl"

In [ ]:
def create_datasets(data, subcorpora):
    small_text_datasets = {}
    for key in subcorpora:
        train = data[key]["init_plus_train"]["text"].to_list()
        train_labels = np.array(data[key]["init_plus_train"]["label_binary"])
        test = data[key]["test"]["text"].to_list()
        test_labels = np.array(data[key]["test"]["label_binary"])
        small_text_datasets[key] = {
            'smalltext_train_dset': TextDataset.from_arrays(train, y = train_labels, target_labels= np.array([0,1])),
            'smalltext_test_dset': TextDataset.from_arrays(test, y = test_labels, target_labels= np.array([0,1]))
        }

    # Save to disk
    with open('../../data/active_learning/small_text_datasets.pkl', "wb") as f:
        pickle.dump(small_text_datasets, f)


In [19]:
create_datasets(datasets, ['psyB', 'psyC'])